# 🧬 Project 5 — Semantic Correspondence
**Team:** Johnprice Osagie · Mario Lapadula · Giorgia Pugliese · Riccardo Bellanca

## 📦 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -b main https://github.com/Johnnyprice29/Project_AML.git /content/Project_AML
%cd /content/Project_AML
!git fetch origin && git reset --hard origin/main
!pip install -r requirements.txt -q
!pip install gradio -q
!python dataloaders/download_spair.py --root ./data

import os, torch, gc
from utils.demo_utils import launch_stage_demo, launch_comparison_demo, launch_robustness_demo
DRIVE_CKPTS = '/content/drive/MyDrive/AML/Checkpoints'

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print('[INFO] GPU Memory cleared.')

## 🔍 1.1 Evaluate Backbones (Baseline Analysis)

In [ ]:
# 1.1.1 DINOv2
clear_gpu()
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --backbone dinov2_vitb14 --results_file /content/drive/MyDrive/AML/Results/baseline_dinov2.txt

In [ ]:
# 1.1.2 DINOv3 (Reg)
clear_gpu()
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --backbone dinov3 --results_file /content/drive/MyDrive/AML/Results/baseline_dinov3.txt

In [ ]:
# 1.1.3 SAM (ViT-B)
clear_gpu()
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --backbone sam_vitb --batch_size 1 --results_file /content/drive/MyDrive/AML/Results/baseline_sam.txt

## 🔦 1.2 Layer-wise Explorer for DINOv2

In [ ]:
launch_stage_demo('DINOv2 Layer Explorer', show_layer_slider=True)

## 🚀 2. Training DINOv2 (LoRA, LoRA+Curriculum)

In [ ]:
# 2.1 Training LoRA
if not os.path.exists(f'{DRIVE_CKPTS}/lora_only/lora_only_best.pth'):
    !python train.py --peft_type lora --dataset_root ./data/SPair-71k --epochs 5 --exp_name lora_only --output_dir ./checkpoints/lora_only --backup_dir $DRIVE_CKPTS/lora_only
else: print('[OK] LoRA già presente.')

## 🎯 3. Raffinamento (Ablation Study: Adaptive Window)

In [ ]:
CKPT_LORA = f'{DRIVE_CKPTS}/lora_only/lora_only_best.pth'
if os.path.exists(CKPT_LORA):
    !python evaluate.py --dataset_root ./data/SPair-71k --checkpoint $CKPT_LORA --results_file /content/drive/MyDrive/AML/Results/lora_final_aw.txt
else:
    print(f'[ERROR] Checkpoint non trovato in {CKPT_LORA}. Assicurati di aver completato lo Stage 2.')

## ⚖️ 5. Showcase Finali

In [ ]:
launch_comparison_demo(ckpt_name='lora_only')

In [ ]:
launch_robustness_demo(ckpt_name='lora_only')